# IE-OPT-08 · 铝材下料

**场景**：北麓门窗厂把长度 16 分米的标准铝材锯成三种窗料。本周订单是立挺 6 分米 6 根、横档 5 分米 5 根、压条 4 分米 7 根。一根母材只能按一种裁法切开，问至少要用多少根母材。

裁法先在小宽度上全部列出来，再决定每种裁法用几次。工业规模的下料会用列生成，因为裁法数量会随规格变多；这个例子宽度只有 16，枚举裁法是可行的。数据为合成教学数据。

练习用，不计正式成绩。

## 1. 模型要点

- 先列出「不能再塞进任何一种窗料」的裁法。对覆盖模型来说，这样做不改变最少母材数：没裁满的方案总能再放一根料，母材根数不会增加。
- $y_p$ 是非负整数，表示裁法 $p$ 使用的母材根数。
- 最小化 $\sum_p y_p$。
- 对每种窗料，各裁法产出的根数之和不少于订单。

允许某种料多切出来。模型不管锯缝、余料回收价和交货顺序。

In [ ]:
from pyscipopt import Model, quicksum

WIDTH = 16
piece_names = ["立挺", "横档", "压条"]
lengths = [6, 5, 4]
demands = [6, 5, 7]
assert all(0 < length <= WIDTH for length in lengths)

def maximal_patterns(width, lengths):
    found = []

    def rec(index, left, counts):
        if index == len(lengths):
            if width - left > 0 and left < min(lengths):
                found.append(tuple(counts))
            return
        room = left // lengths[index]
        for count in range(room + 1):
            rec(index + 1, left - count * lengths[index], counts + (count,))

    rec(0, width, ())
    return found

patterns = maximal_patterns(WIDTH, lengths)
print("可用裁法数:", len(patterns))
for pattern in patterns:
    used = sum(length * count for length, count in zip(lengths, pattern))
    print(dict(zip(piece_names, pattern)), "用去", used, "分米")

## 2. 用母材根数做目标

In [ ]:
def solve_cutting_stock(patterns, demands):
    model = Model("IE-OPT-08 cutting stock")
    model.hideOutput(True)
    y = {p: model.addVar(vtype="I", lb=0, name=f"rolls_{p}") for p in range(len(patterns))}
    for i, need in enumerate(demands):
        model.addCons(quicksum(patterns[p][i] * y[p] for p in y) >= need)
    model.setObjective(quicksum(y[p] for p in y), "minimize")
    model.optimize()
    status = str(model.getStatus())
    if status != "optimal":
        raise RuntimeError(f"Solver status is {status}; optimality not certified")
    used = {p: int(round(model.getVal(y[p]))) for p in y if model.getVal(y[p]) > 0.5}
    return {"objective": float(model.getObjVal()), "used": used}

result = solve_cutting_stock(patterns, demands)
print("最少母材:", result["objective"])
for index, times in sorted(result["used"].items()):
    print(f"裁法 {patterns[index]} 使用 {times} 根")

## 3. 按剩余订单做动态规划

状态是三种窗料还缺多少。每用一根母材就选一种裁法，把订单减去这次实际切出的数量。状态数是各需求加一的乘积，和 SCIP 的分支过程无关。

In [ ]:
def oracle_min_rolls(demands, patterns):
    cache = {}

    def dp(state):
        if state in cache:
            return cache[state]
        if all(value == 0 for value in state):
            cache[state] = 0
            return 0
        best = 10**6
        for pattern in patterns:
            nxt = []
            useful = False
            for need, cut in zip(state, pattern):
                take = min(need, cut)
                useful = useful or take > 0
                nxt.append(need - take)
            if useful:
                best = min(best, 1 + dp(tuple(nxt)))
        cache[state] = best
        return best

    answer = dp(tuple(demands))
    if answer >= 10**6:
        raise RuntimeError("patterns cannot cover the order")
    return answer

oracle = oracle_min_rolls(demands, patterns)
print("独立动态规划最少母材:", oracle)
assert abs(result["objective"] - oracle) < 1e-6
produced = [0, 0, 0]
for index, times in result["used"].items():
    for i in range(3):
        produced[i] += patterns[index][i] * times
assert all(got >= need for got, need in zip(produced, demands))
print("通过：求解器目标与动态规划一致，订单被覆盖")

## 4. 可以接着改什么

1. 把母材改成 15 分米，看最少根数和余料怎么变。
2. 给余料一个回收收入，目标改成「母材成本减去回收」，注意回收不能高于材料本身。
3. 规格很多时不要再手写全部裁法。那时才需要列生成；本例故意留在可以穷举的规模。